In [1]:
# --------------------------------- Part 1: Imports ---------------------------------
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import time
import pickle
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, auc, accuracy_score)
from scipy.stats import ttest_rel
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, LSTM, Conv1D, Flatten, concatenate, Dropout,
                                     Multiply, Reshape, BatchNormalization, GlobalAveragePooling1D,
                                     Lambda)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [6]:
import warnings
warnings.filterwarnings("ignore")

In [7]:
df1=pd.read_csv("Wednesday-workingHours.pcap_ISCX.csv")
df2=pd.read_csv("Tuesday-WorkingHours.pcap_ISCX.csv")
df3=pd.read_csv("Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv")
df4=pd.read_csv("Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv")
df5=pd.read_csv("Monday-WorkingHours.pcap_ISCX.csv")
df6=pd.read_csv("Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
df7=pd.read_csv("Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")
df8=pd.read_csv("Friday-WorkingHours-Morning.pcap_ISCX.csv")
df = pd.concat([df1,df2, df3, df4, df5, df6, df7, df8])
del df1
del df2
del df3
del df4
del df5
del df6
del df7
del df8
print(df.shape)

(2830743, 79)


In [8]:
# Drop columns that are completely homogenous

df_nums = df.select_dtypes(include=[np.number])
print("The columns that are completely homogenous are")
for col in df_nums.columns:
    if (df_nums[col].max()-df_nums[col].min() == 0):
        print(col,)
df.drop([col for col in df_nums.columns if df_nums[col].max()-df_nums[col].min() == 0], axis=1, inplace=True)
print(df.shape)

The columns that are completely homogenous are
 Bwd PSH Flags
 Bwd URG Flags
Fwd Avg Bytes/Bulk
 Fwd Avg Packets/Bulk
 Fwd Avg Bulk Rate
 Bwd Avg Bytes/Bulk
 Bwd Avg Packets/Bulk
Bwd Avg Bulk Rate
(2830743, 71)


In [9]:
# Drop NA rows

df.dropna(axis=0, inplace=True)
print(df.shape)

(2829385, 71)


In [10]:
# Normalization
df_nums = df.select_dtypes(include=[np.number])
df_nums = (df_nums-df_nums.min())/(df_nums.max()-df_nums.min())

for col in df_nums.columns:
    df[col] = df_nums[col]

df.dropna(axis=0, inplace=True)
for c in df.columns:
    print(c + " :", df[c].isna().sum())

 Destination Port : 0
 Flow Duration : 0
 Total Fwd Packets : 0
 Total Backward Packets : 0
Total Length of Fwd Packets : 0
 Total Length of Bwd Packets : 0
 Fwd Packet Length Max : 0
 Fwd Packet Length Min : 0
 Fwd Packet Length Mean : 0
 Fwd Packet Length Std : 0
Bwd Packet Length Max : 0
 Bwd Packet Length Min : 0
 Bwd Packet Length Mean : 0
 Bwd Packet Length Std : 0
Flow Bytes/s : 0
 Flow Packets/s : 0
 Flow IAT Mean : 0
 Flow IAT Std : 0
 Flow IAT Max : 0
 Flow IAT Min : 0
Fwd IAT Total : 0
 Fwd IAT Mean : 0
 Fwd IAT Std : 0
 Fwd IAT Max : 0
 Fwd IAT Min : 0
Bwd IAT Total : 0
 Bwd IAT Mean : 0
 Bwd IAT Std : 0
 Bwd IAT Max : 0
 Bwd IAT Min : 0
Fwd PSH Flags : 0
 Fwd URG Flags : 0
 Fwd Header Length : 0
 Bwd Header Length : 0
Fwd Packets/s : 0
 Bwd Packets/s : 0
 Min Packet Length : 0
 Max Packet Length : 0
 Packet Length Mean : 0
 Packet Length Std : 0
 Packet Length Variance : 0
FIN Flag Count : 0
 SYN Flag Count : 0
 RST Flag Count : 0
 PSH Flag Count : 0
 ACK Flag Count : 0
 U

In [11]:
df.shape

(2827876, 71)

In [12]:
label_counts = df[' Label'].value_counts()
print("Distinct label names and their counts:")
print(label_counts)

Distinct label names and their counts:
BENIGN                        2271320
DoS Hulk                       230124
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name:  Label, dtype: int64


In [13]:
from sklearn.preprocessing import LabelEncoder

# Initialize the label encoder
label_encoder = LabelEncoder()

# Apply label encoding to categorical columns
for col in [' Label']:
    df[col] = label_encoder.fit_transform(df[col])

# Display the first few rows of the encoded DataFrame
print(df.head())

    Destination Port   Flow Duration   Total Fwd Packets  \
0           0.001221        0.000319            0.000000   
1           0.005936        0.000004            0.000046   
2           0.001343        0.000009            0.000041   
3           0.005936        0.000127            0.000073   
4           0.001343        0.000009            0.000036   

    Total Backward Packets  Total Length of Fwd Packets  \
0                 0.000003                 4.651163e-07   
1                 0.000017                 1.333333e-05   
2                 0.000021                 2.441860e-04   
3                 0.000041                 2.675969e-04   
4                 0.000021                 2.441860e-04   

    Total Length of Bwd Packets   Fwd Packet Length Max  \
0                  9.153974e-09                0.000242   
1                  4.973659e-07                0.003183   
2                  4.805836e-06                0.063457   
3                  1.016091e-05                0

In [14]:
df.columns = df.columns.str.strip()

selected_features = [  # selected features by IMOA
    'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Std',
    'Bwd Packet Length Min', 'Bwd Packet Length Std', 'Flow IAT Std', 'Flow IAT Max',
    'Flow IAT Min', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Mean', 'Bwd IAT Max',
    'Bwd IAT Min', 'Fwd URG Flags', 'Fwd Header Length', 'Bwd Header Length',
    'Bwd Packets/s', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Variance',
    'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count',
    'URG Flag Count', 'CWE Flag Count', 'Down/Up Ratio', 'Average Packet Size',
    'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Subflow Fwd Packets',
    'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Active Mean', 'Active Max',
    'Active Min', 'Idle Std'
]

In [15]:
X = df[selected_features].values
y = np.where(df['Label'].values == 0, 0, 1)  # Binary classification: 0 = benign, 1 = attack

In [16]:
X.shape

(2827876, 40)

In [17]:

# --------------------------------- Part 3: Define Models ---------------------------------
def create_cnn_model(input_shape):
    inputs = Input(shape=input_shape)
    x = Conv1D(64, 3, activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid')(x)
    return Model(inputs, output)

def create_lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    x = LSTM(64, return_sequences=True)(inputs)
    x = BatchNormalization()(x)
    x = LSTM(32)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid')(x)
    return Model(inputs, output)

def create_fnn_model(input_shape):
    inputs = Input(shape=(input_shape,))
    x = Dense(128, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid')(x)
    return Model(inputs, output)

def attention_mechanism(inputs):
    attention_weights = Dense(inputs.shape[-1], activation='softmax', name='attention_weights')(inputs)
    attention_output = Multiply(name='attention_output')([inputs, attention_weights])
    return attention_output, attention_weights

In [18]:
from memory_profiler import memory_usage

In [19]:
def train_ensemble():
    return ensemble_model.fit(
        [X_train_cnn, X_train_cnn, X_train], y_train,
        epochs=50,
        batch_size=128,
        validation_split=0.1,
        verbose=0,
        validation_data=([X_test_cnn, X_test_cnn, X_test], y_test)
    )

In [24]:

# --------------------------------- Part 4: 5-Fold Cross-Validation Setup ---------------------------------
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

ensemble_accs, cnn_accs, lstm_accs, fnn_accs = [], [], [], []
ensemble_times, cnn_times, lstm_times, fnn_times = [], [], [], []
ensemble_memory_usages, cnn_memory_usages, lstm_memory_usages, fnn_memory_usages = [], [], [], []
all_y_test = []
all_ensemble_pred = []
all_cnn_pred =[]
all_lstm_pred=[]
all_fnn_pred=[]
all_ensemble_prob = []
all_cm = []
fold = 1
for train_idx, test_idx in kfold.split(X, y):
    print(f"\n=== Fold {fold} ===")
    fold += 1

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    # CNN
    cnn_model = create_cnn_model((X_train_cnn.shape[1], 1))
    cnn_model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    start = time.time()
    cnn_mem_usage, cnn_history = memory_usage(
    (cnn_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True )
    #cnn_history = cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    cnn_times.append(end - start)
    cnn_peak_memory = max(cnn_mem_usage)
    cnn_memory_usages.append(cnn_peak_memory)
    cnn_pred = (cnn_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    cnn_accs.append(accuracy_score(y_test, cnn_pred))
    all_cnn_pred.append(cnn_pred)

    # LSTM
   
    lstm_model = create_lstm_model((X_train_cnn.shape[1], 1))
    lstm_model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    start = time.time()
    lstm_mem_usage, lstm_history = memory_usage(
    (lstm_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
     interval=0.1,retval=True)
    #lstm_history= lstm_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    lstm_times.append(end - start)
    lstm_peak_memory = max(lstm_mem_usage)
    lstm_memory_usages.append(lstm_peak_memory)
    lstm_pred = (lstm_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    lstm_accs.append(accuracy_score(y_test, lstm_pred))
    all_lstm_pred.append(lstm_pred)

    # FNN

    fnn_model = create_fnn_model(X_train.shape[1])
    fnn_model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    start = time.time()
    fnn_mem_usage, fnn_history = memory_usage(
    (fnn_model.fit, (X_train, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True)
    #fnn_history=fnn_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    fnn_times.append(end - start)
    fnn_peak_memory = max(fnn_mem_usage)
    fnn_memory_usages.append(fnn_peak_memory)
    fnn_pred = (fnn_model.predict(X_test,verbose=0) > 0.5).astype(int)
    fnn_accs.append(accuracy_score(y_test, fnn_pred))
    all_fnn_pred.append(fnn_pred)

    # Ensemble

    cnn_out = Lambda(lambda x: x)(cnn_model.output)
    lstm_out = Lambda(lambda x: x)(lstm_model.output)
    fnn_out = Lambda(lambda x: x)(fnn_model.output)
    combined = concatenate([cnn_out, lstm_out, fnn_out])
    combined_dense = Dense(22, activation='relu')(combined)
    attention_output, attention_weights = attention_mechanism(combined_dense)
    final_output = Dense(1, activation='sigmoid')(attention_output)
    ensemble_model = Model(inputs=[cnn_model.input, lstm_model.input, fnn_model.input], outputs=final_output)
    ensemble_model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])

    start = time.time()
    ensemble_mem_usage, ensemble_history = memory_usage(
    train_ensemble,interval=0.1,retval=True)
    ensemble_peak_memory = max(ensemble_mem_usage)
    ensemble_memory_usages.append(ensemble_peak_memory)
    end = time.time()
    ensemble_times.append(end - start)
    ensemble_pred = (ensemble_model.predict([X_test_cnn, X_test_cnn, X_test],verbose=0) > 0.5).astype(int)
    ensemble_accs.append(accuracy_score(y_test, ensemble_pred))
    all_y_test.append(y_test)
    all_ensemble_pred.append(ensemble_pred)
    all_ensemble_prob.append(ensemble_model.predict([X_test_cnn, X_test_cnn, X_test],verbose=0))  # Probabilities for ROC/PR
    cm = confusion_matrix(y_test, ensemble_pred)
    all_cm.append(cm)



=== Fold 1 ===


=== Fold 2 ===

=== Fold 3 ===

=== Fold 4 ===

=== Fold 5 ===


In [45]:

# --------------------------------- Part 5: Report Results ---------------------------------
def report_scores(name, scores, times, memories):
    print(f"{name}: Accuracy = {np.mean(scores):.4f} ± {np.std(scores):.4f}, "
          f"Time = {np.mean(times):.2f}s ± {np.std(times):.2f}s, "
          f"Memory = {np.mean(memories):.2f} MiB ± {np.std(memories):.2f} MiB")

print("\n=== 5-Fold Cross-validation Results ===")
report_scores("CNN", cnn_accs, cnn_times, cnn_memory_usages)
report_scores("LSTM", lstm_accs, lstm_times, lstm_memory_usages)
report_scores("FNN", fnn_accs, fnn_times, fnn_memory_usages)
report_scores("Ensemble", ensemble_accs, ensemble_times, ensemble_memory_usages)


=== 5-Fold Cross-validation Results ===
CNN: Accuracy = 0.8808 ± 0.0068, Time = 7792.06s ± 316.74s, Memory = 1786.77 MiB ± 283.10 MiB
LSTM: Accuracy = 0.8835 ± 0.0022, Time = 54275.17s ± 4234.17s, Memory = 1836.07 MiB ± 294.74 MiB
FNN: Accuracy = 0.8960 ± 0.0018, Time = 3613.39s ± 195.87s, Memory = 1613.43 MiB ± 127.93 MiB
Ensemble: Accuracy = 0.9121 ± 0.0047, Time = 52948.40s ± 2053.89s, Memory = 1709.12 MiB ± 115.38 MiB


In [46]:
# --------------------------------- Part 6: Statistical Significance Testing ---------------------------------
print("\n=== Paired t-tests ===")
print("Ensemble vs CNN:", ttest_rel(ensemble_accs, cnn_accs))
print("Ensemble vs LSTM:", ttest_rel(ensemble_accs, lstm_accs))
print("Ensemble vs FNN:", ttest_rel(ensemble_accs, fnn_accs))


=== Paired t-tests ===
Ensemble vs CNN: TtestResult(statistic=7.206359649712708, pvalue=0.001965599396494639, df=4)
Ensemble vs LSTM: TtestResult(statistic=15.211474861701399, pvalue=0.00010890699161632755, df=4)
Ensemble vs FNN: TtestResult(statistic=4.996002253511478, pvalue=0.007511655583612574, df=4)


In [47]:

# Stack all test labels and predictions
y_true = np.concatenate(all_y_test)
y_pred_ensemble = np.concatenate(all_ensemble_pred)
y_prob_ensemble = np.concatenate(all_ensemble_prob)

In [50]:
# --- Classification Report ---
print("\nAblation study-Average Classification Report CIC-IDS2017:(without WGAN-GP)")
print(classification_report(y_true, y_pred_ensemble))


Ablation study-Average Classification Report CIC-IDS2017:(without WGAN-GP)
              precision    recall  f1-score   support

           0       0.99      0.89      0.94  11301605
           1       0.69      0.98      0.81   2837772

    accuracy                           0.91  14139377
   macro avg       0.84      0.93      0.87  14139377
weighted avg       0.93      0.91      0.91  14139377

